In [12]:
# =============================================================================
# BLOQUE 0.1: INGESTIÓN ROBUSTA (KAGGLER RECURSIVO)
# =============================================================================
import kagglehub
import pandas as pd
import glob
import os

try:
    # 1. Descarga (Ignoramos el warning de versión por ahora para avanzar)
    path = kagglehub.dataset_download("jocelyndumlao/cardiovascular-disease-dataset")

    # 2. Búsqueda profunda de cualquier archivo CSV en la descarga
    csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

    if csv_files:
        df = pd.read_csv(csv_files[0]) # Cargamos el primer CSV hallado
        # 3. Persistencia Global
        %store df
        print(f"✅ DATASET IDENTIFICADO: {os.path.basename(csv_files[0])}")
        print(f"📊 DIMENSIONES: {df.shape[0]} pacientes / {df.shape[1]} variables.")
    else:
        print("❌ ERROR: El dataset se descargó pero no contiene archivos .csv")
except Exception as e:
    print(f"❌ ERROR CRÍTICO: {e}")

# =============================================================================
# BLOQUE 1: ENCABEZADO DE FASE Y REPORTE DE INGESTIÓN (CORREGIDO)
# =============================================================================

# 1.1 Recuperación de Datos (Lógica Segura)
if 'df' not in locals():
    try:
        # Intentamos recuperar de la persistencia de Jupyter
        get_ipython().run_line_magic('store', '-r df')
    except:
        pass

# 1.2 Verificación y Renderizado
if 'df' in locals():
    reg, var = df.shape
    accent = "#6366f1" # Indigo Slate unificado

    header_ingestion_html = f"""
    <div style="font-family:'Inter',sans-serif; max-width:1100px; margin:20px auto;">
        <!-- Título de Fase -->
        <div style="display:flex; align-items:center; gap:15px; margin: 40px 0 20px 0;">
            <div style="background:{accent}; color:white; width:40px; height:40px; border-radius:50%; display:flex; align-items:center; justify-content:center; font-family:'Montserrat'; font-weight:800; font-size:1.1em;">02</div>
            <h2 style="font-family:'Montserrat'; font-size:1.6em; font-weight:800; text-transform:uppercase; letter-spacing:1.5px; color:{accent}; margin:0;">Data Understanding</h2>
        </div>

        <!-- Card de Auditoría de Ingestión -->
        <div style="background:#fff; border-radius:15px; padding:30px; box-shadow:0 10px 35px rgba(0,0,0,0.03); border:1px solid #eef2f3;">
            <h4 style="margin-top:0; color:{accent}; font-family:'Montserrat'; font-weight:800; text-transform:uppercase; letter-spacing:2px; font-size:0.85em; border-bottom:1px solid #f1f5f9; padding-bottom:15px; display:flex; align-items:center; gap:10px;">
                <span>📊</span> ESTADO DE LA INGESTIÓN DE DATOS (F2.1)
            </h4>
            <table style="width:100%; border-collapse:collapse; color:#475569; margin-top:10px; font-size:0.95em;">
                <tr><td style="padding:14px 0; border-bottom:1px solid #f8fafc; font-weight:600;">Registros Totales:</td><td style="text-align:right; color:{accent}; font-weight:800; font-family:'Montserrat';">{reg:,} pacientes</td></tr>
                <tr><td style="padding:14px 0; border-bottom:1px solid #f8fafc; font-weight:600;">Variables Clínicas:</td><td style="text-align:right; color:{accent}; font-weight:800; font-family:'Montserrat';">{var} dimensiones</td></tr>
                <tr><td style="padding:14px 0; border-bottom:1px solid #f8fafc; font-weight:600;">Persistencia:</td><td style="text-align:right; color:#10b981; font-weight:700;">✔ Recuperado via %store -r df</td></tr>
                <tr><td style="padding:14px 0; font-weight:600;">Seguridad:</td><td style="text-align:right;"><span style="background:#f5f3ff; color:{accent}; padding:5px 12px; border-radius:6px; font-weight:800; font-size:0.65em; border:1px solid #e0e7ff;">KAGGLE API — REPO-SAFE</span></td></tr>
            </table>
            <div style="margin-top:20px; font-size:0.8em; color:#94a3b8; font-style:italic; border-top:1px solid #f1f5f9; padding-top:15px;">🚀 Los datos han sido vinculados exitosamente desde el Módulo de Carga.</div>
        </div>
    </div>
    """
    display(HTML(header_ingestion_html))
else:
    display(HTML("<div style='color:#ef4444; padding:20px; border:1px solid #fee2e2; border-radius:10px; font-family:sans-serif;'>❌ <b>Error:</b> El DataFrame 'df' no está disponible. Por favor, ejecuta primero la carga de datos.</div>"))

#======================================================================
# BLOQUE 2: EXPLORACIÓN ESTADÍSTICA ESTILIZADA (HEAD & DESCRIBE)
# =============================================================================
def render_premium_table(df_input, title_text, icon="📑"):
    html_raw = df_input.to_html(classes='dataframe')
    template = f"""
    <style>
        .table-wrapper {{ font-family: 'Inter', sans-serif; margin: 30px auto; max-width: 1100px; border-radius: 15px; overflow: hidden; box-shadow: 0 4px 20px rgba(0,0,0,0.04); border: 1px solid #eef2f3; }}
        .table-header-bar {{ background: var(--petroleo); color: white; padding: 12px 20px; font-family: 'Montserrat'; font-weight: 800; font-size: 0.8em; text-transform: uppercase; letter-spacing: 2.5px; display: flex; align-items: center; gap: 10px; }}
        .dataframe {{ width: 100%; border-collapse: collapse; border: none !important; }}
        .dataframe thead th {{ background: #f8fafc; color: var(--petroleo); padding: 12px; font-family: 'Montserrat'; font-size: 0.72em; text-transform: uppercase; border-bottom: 2px solid #eef2f3 !important; text-align: center; }}
        .dataframe tbody tr:hover {{ background-color: #f8faff; transition: 0.2s; }}
        .dataframe td {{ padding: 10px; font-size: 0.88em; color: #475569; border-bottom: 1px solid #f1f5f9; text-align: center; }}
        .dataframe tbody th {{ background: #fdfaff; color: var(--indigo-slate); font-weight: 700; border-right: 2px solid #f1f5f9; padding: 10px; font-size: 0.82em; }}
    </style>
    <div class="table-wrapper">
        <div class="table-header-bar"><span>{icon}</span> {title_text} (F2.2)</div>
        {html_raw}
    </div>
    """
    return HTML(template)

# Renderizado de Exploración
display(render_premium_table(df.head(), "Vista Previa de Registros", icon="📋"))
display(render_premium_table(df.describe().round(2), "Distribución Estadística Global", icon="📈"))

# #############################################################################
# DICCIONARIO CLÍNICO V.10: CORRECCIÓN ESTRUCTURAL (FIX restingBP)
# #############################################################################
from IPython.display import HTML, display

dict_v10_html = """
<style>
    @import url('https://fonts.googleapis.com');

    .dict-v10 { font-family: 'Inter', sans-serif; max-width: 1100px; margin: 30px auto; background: #fff; padding: 25px; border-radius: 20px; border: 1px solid #f1f5f9; }
    .header-box { border-bottom: 2px solid #2e6171; padding-bottom: 15px; margin-bottom: 25px; display: flex; align-items: center; justify-content: space-between; }
    .header-box h2 { font-family: 'Montserrat'; color: #2e6171; font-size: 1.15em; margin: 0; text-transform: uppercase; letter-spacing: 2px; }

    .t-bar { display: flex; background: #2e6171; color: white; padding: 14px 20px; border-radius: 8px; font-family: 'Montserrat'; font-size: 0.72em; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 12px; }
    .d-row { display: flex; align-items: center; padding: 12px 20px; margin-bottom: 5px; background: #fff; border-radius: 10px; border: 1px solid #f8fafc; transition: 0.3s cubic-bezier(0.4, 0, 0.2, 1); }

    /* Columnas: Tipo 12% | Nombre 18% | Significado 40% | Valores 30% */
    .w-type { width: 12%; } .w-name { width: 18%; font-weight: 700; color: #2e6171; font-family: 'Montserrat'; font-size: 0.85em; }
    .w-desc { width: 40%; color: #64748b; font-size: 0.9em; padding: 0 15px; line-height: 1.4; } .w-vals { width: 30%; display: flex; flex-wrap: wrap; gap: 8px; }

    .tag-pill, .tag-type { padding: 4px 12px; border-radius: 6px; font-size: 0.72em; font-weight: 600; border: 1px solid #f1f5f9; color: #cbd5e1; background: transparent; transition: 0.3s; }
    .tag-type { text-transform: uppercase; font-size: 0.58em; font-weight: 800; border-style: dashed; }

    /* Efectos Hover */
    .row-cat:hover { border-color: #10b981; background: #f0fdf4; transform: translateX(8px); }
    .row-cat:hover .tag-pill, .row-cat:hover .tag-type { background: #10b981; color: white; border-style: solid; border-color: #10b981; }

    .row-bin:hover { border-color: #f59e0b; background: #fffbeb; transform: translateX(8px); }
    .row-bin:hover .tag-pill, .row-bin:hover .tag-type { background: #f59e0b; color: white; border-style: solid; border-color: #f59e0b; }

    .row-num:hover { border-color: #6366f1; background: #eff6ff; transform: translateX(8px); }
    .row-num:hover .tag-pill, .row-num:hover .tag-type { background: #6366f1; color: white; border-style: solid; border-color: #6366f1; }

    .row-target:hover { border-color: #8b5cf6; background: #f5f3ff; transform: scale(1.01); }
    .row-target:hover .tag-pill, .row-target:hover .tag-type { background: #8b5cf6; color: white; border-style: solid; border-color: #8b5cf6; }
</style>

<div class="dict-v10">
    <div class="header-box"><h2>📖 Diccionario Clínico: Auditoría Isquémica</h2><span>F2.3 • FINAL REFINED</span></div>
    <div class="t-bar"><div style="width:12%;">Tipo</div><div style="width:18%;">Variable</div><div style="width:40%;">Significado Clínico</div><div style="width:30%;">Valores Posibles</div></div>

    <div class="d-row row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">gender</div><div class="w-desc">Género biológico del paciente analizado.</div><div class="w-vals"><span class="tag-pill">0: Mujer</span><span class="tag-pill">1: Hombre</span></div></div>
    <div class="d-row row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">chestpain</div><div class="w-desc">Tipo de dolor torácico percibido subjetivamente.</div><div class="w-vals"><span class="tag-pill">0: Típico</span><span class="tag-pill">1: Atípico</span><span class="tag-pill">2: No-anginal</span><span class="tag-pill">3: Asín.</span></div></div>
    <div class="d-row row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">restingrelectro</div><div class="w-desc">Resultados de ECG en estado de reposo absoluto.</div><div class="w-vals"><span class="tag-pill">0: Normal</span><span class="tag-pill">1: ST-T</span><span class="pill tag-pill">2: HVI</span></div></div>
    <div class="d-row row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">slope</div><div class="w-desc">Pendiente del segmento ST en el pico del ejercicio.</div><div class="w-vals"><span class="tag-pill">1: Asc</span><span class="tag-pill">2: Plana</span><span class="tag-pill">3: Desc</span></div></div>

    <div class="d-row row-bin"><div class="w-type"><span class="tag-type">Binario</span></div><div class="w-name">fastingbloodsugar</div><div class="w-desc">Glucemia basal en ayunas mayor a 120 mg/dl.</div><div class="w-vals"><span class="tag-pill">0: No</span><span class="tag-pill">1: Sí</span></div></div>
    <div class="d-row row-bin"><div class="w-type"><span class="tag-type">Binario</span></div><div class="w-name">exerciseangia</div><div class="w-desc">Presencia de angina inducida por esfuerzo físico.</div><div class="w-vals"><span class="tag-pill">0: No</span><span class="tag-pill">1: Sí</span></div></div>

    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">patientid</div><div class="w-desc">Identificador único correlativo del registro.</div><div class="w-vals"><span class="tag-pill">Correlativo ID</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">age</div><div class="w-desc">Edad cronológica del paciente al ingreso.</div><div class="w-vals"><span class="tag-pill">20 - 80 años</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">restingBP</div><div class="w-desc">Presión arterial sistólica en reposo (mmHg).</div><div class="w-vals"><span class="tag-pill">94 - 200 mmHg</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">serumcholestrol</div><div class="w-desc">Colesterol sérico total medido en mg/dl.</div><div class="w-vals"><span class="tag-pill">0 - 602 mg/dl</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">maxheartrate</div><div class="w-desc">Frecuencia cardíaca máxima alcanzada (FCmáx).</div><div class="w-vals"><span class="tag-pill">71 - 202 lpm</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">oldpeak</div><div class="w-desc">Depresión del ST relativa al estado de reposo.</div><div class="w-vals"><span class="tag-pill">0.0 - 6.2 Escala</span></div></div>
    <div class="d-row row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">noofmajorvessels</div><div class="w-desc">Número de vasos mayores por fluoroscopia.</div><div class="w-vals"><span class="tag-pill">0 - 3 Vasos</span></div></div>

    <div class="d-row row-target"><div class="w-type"><span class="tag-type">Target</span></div><div class="w-name" style="color:#8b5cf6;">target</div><div class="w-desc"><b>Diagnóstico clínico de Riesgo Cardiovascular.</b></div><div class="w-vals"><span class="tag-pill">0: Sano</span><span class="tag-pill">1: Riesgo</span></div></div>
</div>
"""
display(HTML(dict_v10_html))

Using Colab cache for faster access to the 'cardiovascular-disease-dataset' dataset.
Stored 'df' (DataFrame)
✅ DATASET IDENTIFICADO: Cardiovascular_Disease_Dataset.csv
📊 DIMENSIONES: 1000 pacientes / 14 variables.


Registros Totales:,"1,000 pacientes"
Variables Clínicas:,14 dimensiones
Persistencia:,✔ Recuperado via %store -r df
Seguridad:,KAGGLE API — REPO-SAFE


,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.0,1000.00,1000.00,1000.00,1000.00
mean,5048704.41,49.24,0.76,0.98,151.75,311.45,0.30,0.75,145.48,0.5,2.71,1.54,1.22,0.58
std,2895904.50,17.86,0.42,0.95,29.97,132.44,0.46,0.77,34.19,0.5,1.72,1.00,0.98,0.49
min,103368.00,20.00,0.00,0.00,94.00,0.00,0.00,0.00,71.00,0.0,0.00,0.00,0.00,0.00
25%,2536439.50,34.00,1.00,0.00,129.00,235.75,0.00,0.00,119.75,0.0,1.30,1.00,0.00,0.00
50%,4952508.50,49.00,1.00,1.00,147.00,318.00,0.00,1.00,146.00,0.0,2.40,2.00,1.00,1.00
75%,7681877.00,64.25,1.00,2.00,181.00,404.25,1.00,1.00,175.00,1.0,4.10,2.00,2.00,1.00
max,9990855.00,80.00,1.00,3.00,200.00,602.00,1.00,2.00,202.00,1.0,6.20,3.00,3.00,1.00


In [66]:
# =============================================================================
# CAJA NEGRA V.3.8: UNIFIED LUMINOUS (BASE V3.7 + DNA F1 SPIN & FORMAT)
# =============================================================================
import pandas as pd
from IPython.display import HTML, display

# 1. MOTOR DE INGESTIÓN
if 'df' not in locals():
    try:
        get_ipython().run_line_magic('store', '-r df')
    except:
        # Fallback técnico por si el store falla en el test
        df = pd.DataFrame({'Métrica': ['ST_Slope', 'Recall'], 'Valor': [0.80, 0.85]})

# 2. ESTILOS MAESTROS (DNA F1 INJECTED: SPINNING BORDERS)
f2_styles_v38 = """
<style>
    @import url('https://fonts.googleapis.com');
    :root {
        --f2-violet: #8b5cf6; --arctic: #f8fafc; --data-gray: #cbd5e1;
        --cyan: #00f2fe; --red: #ff4444; --green: #00ff88;
    }

    /* REGISTRO DE PROPIEDAD PARA ANIMACIÓN (HERENCIA F1) */
    @property --angle { syntax: '<angle>'; initial-value: 0deg; inherits: false; }
    @keyframes spin { from { --angle: 0deg; } to { --angle: 360deg; } }

    .master-container {
        font-family: 'Inter', sans-serif; max-width: 950px; margin: 40px auto;
        background: rgba(10, 20, 30, 0.75); backdrop-filter: blur(30px);
        border-radius: 35px; border: 1px solid rgba(139, 92, 246, 0.2);
        padding: 45px; color: white; transform-style: preserve-3d;
        box-shadow: 0 50px 100px rgba(0,0,0,0.8); position: relative; perspective: 1500px;
    }

    .reveal-group { display: flex; gap: 12px; margin-bottom: 35px; position: relative; z-index: 10; }
    .reveal-btn {
        padding: 10px 18px !important; border-radius: 50px;
        background: rgba(255, 255, 255, 0.05); border: 1px solid rgba(255, 255, 255, 0.15);
        color: white !important; font-size: 0.75em; cursor: pointer;
        transition: all 0.6s cubic-bezier(0.165, 0.84, 0.44, 1);
        display: inline-flex; align-items: center; overflow: hidden;
        max-width: 115px; min-width: 115px; white-space: nowrap; font-weight: 600; text-decoration: none;
    }
    .reveal-btn:hover { max-width: 450px !important; background: rgba(255, 255, 255, 0.1) !important; transform: translateY(-3px); }
    .reveal-btn strong { font-family: 'Orbitron'; color: var(--cyan); flex-shrink: 0; margin-right: 15px; }
    .reveal-btn:hover strong { color: var(--f2-violet) !important; }
    .reveal-btn span { opacity: 0; transition: 0.4s; pointer-events: none; color: #fff !important; font-family: 'JetBrains Mono', monospace; }
    .reveal-btn:hover span { opacity: 1; }

    /* FILAS DE FASE CON GIRO DINÁMICO (INYECTADO) */
    .phase-row {
        position: relative; overflow: hidden; margin-bottom: 12px;
        background: rgba(255, 255, 255, 0.03); border: 1px solid rgba(255, 255, 255, 0.07);
        transition: all 0.4s cubic-bezier(0.165, 0.84, 0.44, 1); border-radius: 20px;
        padding: 25px 35px; display: flex; align-items: center; z-index: 2;
        --p-color: var(--f2-violet);
    }
    .phase-row:hover { transform: translateX(8px) translateZ(15px); }
    .phase-row::before {
        content: ''; position: absolute; inset: 0; padding: 2px; border-radius: inherit;
        background: conic-gradient(from var(--angle), transparent 75%, var(--p-color) 85%, var(--p-color) 95%, transparent 100%);
        -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
        mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
        -webkit-mask-composite: xor; mask-composite: exclude; opacity: 0; transition: 0.3s;
    }
    .phase-row:hover::before { opacity: 1; animation: spin 1.5s linear infinite; }
    .f-green { --p-color: var(--green) !important; }
    .f-red { --p-color: var(--red) !important; }

    .phase-label { font-family: 'Orbitron'; font-size: 0.8em; width: 135px; font-weight: 900; color: #666; transition: 0.4s; }
    .phase-row:hover .phase-label { color: var(--p-color); text-shadow: 0 0 15px var(--p-color); }

    .audit-section { margin: 25px 0; border-top: 1px solid rgba(139, 92, 246, 0.1); padding-top: 20px; }
    .audit-row { display: flex; justify-content: space-between; padding: 12px 20px; position: relative; transition: 0.4s; border-radius: 10px; }
    .audit-row::before { content: ''; position: absolute; left: 0; bottom: 0; width: 0; height: 1px; background: var(--f2-violet); box-shadow: 0 0 10px var(--f2-violet); transition: 0.6s; }
    .audit-row:hover::before { width: 100%; }
    .audit-row:hover { background: rgba(255, 255, 255, 0.02); }

    .obj-divider { display: flex; align-items: center; text-align: center; margin: 35px 0 25px 0; font-family: 'Orbitron'; font-size: 0.85em; letter-spacing: 6px; color: #555; font-weight: 900; }
    .obj-divider::before, .obj-divider::after { content: ''; flex: 1; border-bottom: 1px solid rgba(255,255,255,0.08); margin: 0 20px; }

    .glass-wrapper { font-family: 'Inter', sans-serif; margin: 40px auto; max-width: 1100px; background: #050a0f; border-radius: 20px; border: 1px solid rgba(139, 92, 246, 0.15); overflow: hidden; }
    .table-container { overflow-x: scroll !important; width: 100%; position: relative; scrollbar-width: thin; scrollbar-color: var(--f2-violet) #111; }
    .glass-df { min-width: 1800px; border-collapse: separate; border-spacing: 0; color: var(--data-gray); font-size: 0.85em; }
    .glass-df thead th { padding: 25px 40px; background: #000; color: #94a3b8; font-family: 'Orbitron'; font-weight: 800; font-size: 0.65em; text-transform: uppercase; border-bottom: 1px solid rgba(139, 92, 246, 0.3) !important; white-space: nowrap; }
    .glass-df td { padding: 15px 40px; text-align: center; border-bottom: 1px solid rgba(255,255,255,0.03); border-right: 1px solid rgba(255,255,255,0.02); transition: color 0.3s; }
    .glass-df td:hover { background: var(--arctic) !important; color: #000 !important; font-weight: 900; box-shadow: 0 0 25px rgba(255, 255, 255, 0.4); z-index: 10; }
    .glass-df tbody th { background: rgba(0,0,0,0.9); color: #94a3b8; font-weight: 700; padding: 12px 25px; font-size: 0.75em; border-right: 1.5px solid var(--f2-violet); position: sticky; left: 0; z-index: 5; }
</style>
"""

# 3. MOTOR DE RENDERIZADO TABULAR (V3.7 BASE)
def render_v38(df_input, title_text, step_id="F2.2"):
    html_raw = df_input.to_html(classes='glass-df')
    u_id = f"v38_{step_id.replace('.', '_')}"
    return f"""
    <div class="glass-wrapper">
        <div class="glass-header" style="background:rgba(0,0,0,0.7); padding:18px 25px; border-bottom:1px solid rgba(139,92,246,0.1); display:flex; justify-content:space-between; align-items:center;">
            <span style="font-family:'Orbitron'; color:var(--arctic); font-weight:800; font-size:0.75em; letter-spacing:2px; display:flex; align-items:center; gap:10px;">
                <span style="color:var(--f2-violet)">📑</span> {title_text}
            </span>
            <span style="color:var(--f2-violet); font-size:0.6em; font-weight:900; letter-spacing:1px;">{step_id}</span>
        </div>
        <div class="table-container" id="{u_id}">{html_raw}</div>
    </div>
    <script>
    (function() {{
        const table = document.querySelector('#{u_id} .glass-df');
        const cells = table.querySelectorAll('td');
        cells.forEach(cell => {{
            cell.addEventListener('mouseenter', () => {{
                const colIndex = cell.cellIndex;
                const row = cell.parentElement;
                row.querySelectorAll('td').forEach(c => c.style.backgroundColor = 'rgba(255, 255, 255, 0.05)');
                table.querySelectorAll('tr').forEach(r => {{
                    const targetCell = r.cells[colIndex];
                    if(targetCell) targetCell.style.backgroundColor = 'rgba(255, 255, 255, 0.05)';
                }});
            }});
            cell.addEventListener('mouseleave', () => {{
                table.querySelectorAll('td').forEach(c => c.style.backgroundColor = '');
            }});
        }});
    }})();
    </script>
    """

# 4. ENSAMBLAJE FINAL (INCORPORACIÓN MÉTODO .FORMAT())
f2_body_template = """
{styles}
<div class="master-container" id="main-f2-v38">
    <div class="header-section">
        <h1 style="font-family:'Orbitron'; font-weight:900; font-size:2.4em; margin:0; line-height:1.2; color:var(--f2-violet)">Fase 2: Data Understanding</h1>
        <p style="color: #666; font-size: 1.05em; margin-bottom: 35px; font-weight: 600;">Scientific Requirements & Clinical Analysis / IBM Methodology</p>
        <div class="reveal-group">
            <a href="{URL_AUTOR}" target="_blank" class="reveal-btn"><strong>AUTOR</strong> <span>/ juanjararPYBM</span></a>
            <a href="{URL_INDEX}" target="_blank" class="reveal-btn"><strong>INDEX</strong> <span>/ Master Portfolio</span></a>
            <a href="#f2-data-section" class="reveal-btn" style="--p-color: var(--green)"><strong>DATA</strong> <span>/ Exploratory Analysis</span></a>
        </div>
    </div>

    <div class="audit-section">
        <div class="audit-header" style="font-family:'Orbitron'; font-size:0.75em; letter-spacing:3px; color:#94a3b8; margin-bottom:15px;"><b>■</b> ESTADO DE LA INGESTIÓN DE DATOS (F2.1)</div>
        <div class="audit-row"><span style="color:#64748b; font-size:0.9em;">Registros Totales:</span><span style="font-family:'JetBrains Mono'; font-weight:700;">{reg} pacientes</span></div>
        <div class="audit-row"><span style="color:#64748b; font-size:0.9em;">Variables Clínicas:</span><span style="font-family:'JetBrains Mono'; font-weight:700;">{var} dimensiones</span></div>
        <div class="audit-row"><span style="color:#64748b; font-size:0.9em;">Persistencia:</span><span style="color:#00ff88; font-family:'JetBrains Mono'; font-size:0.85em;">✔ Recuperado via %store -r df</span></div>
    </div>

    <div class="obj-divider">CONCEPTOS Y REQUERIMIENTOS</div>
    <div class="phase-row f-green"><div class="phase-label">ST-SLOPE</div><div class="phase-name"><b>Métrica Maestra (0.80):</b> El análisis electrofisiológico identifica la pendiente descendente como signo patognomónico de isquemia severa.</div></div>
    <div class="phase-row"><div class="phase-label">THRESHOLDS</div><div class="phase-name"><b>Umbrales AHA/ESC:</b> Definición de límites clínicos (Presión >140 / Colesterol >240) para disparar la <b>Alerta Temprana</b>.</div></div>
    <div class="phase-row f-red"><div class="phase-label">SILENT RISK</div><div class="phase-name"><b>Isquemia Silente:</b> El 11.3% de riesgo asintomático en mujeres justifica optimizar el <b>Recall</b> para no ignorar el riesgo oculto.</div></div>
</div>

<div id="f2-data-section" style="margin-top: 40px;">
    {table_preview}
    {table_stats}
</div>

<!-- EL DICCIONARIO AHORA CIERRA LA FASE -->
    <div class="obj-divider">DICCIONARIO CLÍNICO V.10.1</div>
    {diccionario}
</div>


<script src="https://cdnjs.cloudflare.com"></script>
<script>
    (function() {{
        const card = document.getElementById('main-f2-v38');
        if(card) {{
            document.addEventListener('mousemove', (e) => {{
                let x = (window.innerWidth / 2 - e.clientX) / 150;
                let y = (window.innerHeight / 2 - e.clientY) / 150;
                gsap.to(card, {{ duration: 0.8, rotationY: x, rotationX: -y, ease: "power1.out" }});
            }});
        }}
    }})();
</script>
"""
# #############################################################################
# DICCIONARIO CLÍNICO V.10.1: FULL DATA INTEGRITY (LEGACY TEXT REMOVED)
# #############################################################################
from IPython.display import HTML, display

dict_v10_final = """
<style>
    @import url('https://fonts.googleapis.com');
    :root {
        --f2-violet: #8b5cf6; --arctic: #f8fafc; --cyan: #00f2fe;
        --red: #ff4444; --green: #00ff88; --dark-bg: #0a141e;
    }

    @property --angle { syntax: '<angle>'; initial-value: 0deg; inherits: false; }
    @keyframes spin { from { --angle: 0deg; } to { --angle: 360deg; } }

    .dict-v10-standalone {
        font-family: 'Inter', sans-serif; max-width: 1000px; margin: 30px auto;
        background: var(--dark-bg); padding: 35px; border-radius: 30px;
        border: 1px solid rgba(139, 92, 246, 0.15); color: white;
    }

    /* CABECERA LIMPIA: SOLO TÍTULO ÁRTICO */
    .header-box-v10 { border-bottom: 1px solid rgba(255,255,255,0.05); padding-bottom: 20px; margin-bottom: 25px; display: flex; align-items: center; }
    .header-box-v10 h2 { font-family: 'Orbitron'; color: var(--arctic); font-size: 1.1em; margin: 0; text-transform: uppercase; letter-spacing: 2px; font-weight: 900; }

    .t-bar-v10 {
        position: relative; display: flex; background: rgba(139, 92, 246, 0.05);
        color: var(--f2-violet); padding: 14px 20px; border-radius: 12px;
        font-family: 'Orbitron'; font-size: 0.65em; text-transform: uppercase;
        letter-spacing: 2px; margin-bottom: 15px; overflow: hidden;
    }
    .t-bar-v10::before {
        content: ''; position: absolute; inset: 0; padding: 1px; border-radius: inherit;
        background: conic-gradient(from var(--angle), transparent 80%, var(--f2-violet) 90%, transparent 100%);
        -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
        mask-composite: exclude; animation: spin 3s linear infinite;
    }

    .d-row-v10 { display: flex; align-items: center; padding: 14px 20px; margin-bottom: 8px; background: rgba(255,255,255,0.02); border-radius: 15px; border: 1px solid rgba(255,255,255,0.05); transition: 0.4s cubic-bezier(0.165, 0.84, 0.44, 1); }
    .w-type { width: 12%; } .w-name { width: 18%; font-weight: 800; color: var(--arctic); font-family: 'Orbitron'; font-size: 0.75em; }
    .w-desc { width: 40%; color: #94a3b8; font-size: 0.88em; padding: 0 15px; line-height: 1.5; } .w-vals { width: 30%; display: flex; flex-wrap: wrap; gap: 8px; }

    .tag-pill { padding: 4px 10px; border-radius: 6px; font-size: 0.65em; font-family: 'JetBrains Mono'; background: rgba(255,255,255,0.05); color: #64748b; border: 1px solid rgba(255,255,255,0.1); transition: 0.3s; }
    .tag-type { padding: 3px 8px; border-radius: 4px; font-size: 0.55em; font-weight: 900; text-transform: uppercase; border: 1px dashed rgba(255,255,255,0.2); }

    .row-cat:hover { border-color: var(--green); background: rgba(0, 255, 136, 0.05); transform: translateX(8px); }
    .row-cat:hover .tag-pill { background: var(--green); color: black; border-color: var(--green); }
    .row-bin:hover { border-color: #f59e0b; background: rgba(245, 158, 11, 0.05); transform: translateX(8px); }
    .row-bin:hover .tag-pill { background: #f59e0b; color: black; border-color: #f59e0b; }
    .row-num:hover { border-color: var(--cyan); background: rgba(0, 242, 254, 0.05); transform: translateX(8px); }
    .row-num:hover .tag-pill { background: var(--cyan); color: black; border-color: var(--cyan); }

    .row-target { border-color: rgba(255, 68, 68, 0.1); }
    .row-target:hover { border-color: var(--red); background: rgba(255, 68, 68, 0.08); transform: scale(1.02); box-shadow: 0 0 20px rgba(255, 68, 68, 0.15); }
    .row-target .tag-type { color: var(--red); border-color: var(--red); }
    .row-target:hover .tag-pill { background: var(--red); color: white; border-color: var(--red); }
</style>

<div class="dict-v10-standalone">
    <div class="header-box-v10"><h2>📖 Diccionario Clínico: Auditoría Isquémica</h2></div>
    <div class="t-bar-v10"><div style="width:12%;">Tipo</div><div style="width:18%;">Variable</div><div style="width:40%;">Significado Clínico</div><div style="width:30%;">Valores Posibles</div></div>

    <div class="d-row-v10 row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">gender</div><div class="w-desc">Género biológico del paciente analizado.</div><div class="w-vals"><span class="tag-pill">0: Mujer</span><span class="tag-pill">1: Hombre</span></div></div>
    <div class="d-row-v10 row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">chestpain</div><div class="w-desc">Tipo de dolor torácico percibido subjetivamente.</div><div class="w-vals"><span class="tag-pill">0: Típico</span><span class="tag-pill">1: Atípico</span><span class="tag-pill">2: No-anginal</span><span class="tag-pill">3: Asín.</span></div></div>
    <div class="d-row-v10 row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">restingrelectro</div><div class="w-desc">Resultados de ECG en estado de reposo absoluto.</div><div class="w-vals"><span class="tag-pill">0: Normal</span><span class="tag-pill">1: ST-T</span><span class="tag-pill">2: HVI</span></div></div>
    <div class="d-row-v10 row-cat"><div class="w-type"><span class="tag-type">Categórico</span></div><div class="w-name">slope</div><div class="w-desc">Pendiente del segmento ST en el pico del ejercicio.</div><div class="w-vals"><span class="tag-pill">1: Asc</span><span class="tag-pill">2: Plana</span><span class="tag-pill">3: Desc</span></div></div>
    <div class="d-row-v10 row-bin"><div class="w-type"><span class="tag-type">Binario</span></div><div class="w-name">fastingbloodsugar</div><div class="w-desc">Glucemia basal en ayunas mayor a 120 mg/dl.</div><div class="w-vals"><span class="tag-pill">0: No</span><span class="tag-pill">1: Sí</span></div></div>
    <div class="d-row-v10 row-bin"><div class="w-type"><span class="tag-type">Binario</span></div><div class="w-name">exerciseangia</div><div class="w-desc">Presencia de angina inducida por esfuerzo físico.</div><div class="w-vals"><span class="tag-pill">0: No</span><span class="tag-pill">1: Sí</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">patientid</div><div class="w-desc">Identificador único correlativo del registro.</div><div class="w-vals"><span class="tag-pill">Correlativo ID</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">age</div><div class="w-desc">Edad cronológica del paciente al ingreso.</div><div class="w-vals"><span class="tag-pill">20 - 80 años</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">restingBP</div><div class="w-desc">Presión arterial sistólica en reposo (mmHg).</div><div class="w-vals"><span class="tag-pill">94 - 200 mmHg</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">serumcholestrol</div><div class="w-desc">Colesterol sérico total medido en mg/dl.</div><div class="w-vals"><span class="tag-pill">0 - 602 mg/dl</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">maxheartrate</div><div class="w-desc">Frecuencia cardíaca máxima alcanzada (FCmáx).</div><div class="w-vals"><span class="tag-pill">71 - 202 lpm</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">oldpeak</div><div class="w-desc">Depresión del ST relativa al estado de reposo.</div><div class="w-vals"><span class="tag-pill">0.0 - 6.2 Escala</span></div></div>
    <div class="d-row-v10 row-num"><div class="w-type"><span class="tag-type">Numérico</span></div><div class="w-name">noofmajorvessels</div><div class="w-desc">Número de vasos mayores por fluoroscopia.</div><div class="w-vals"><span class="tag-pill">0 - 3 Vasos</span></div></div>
    <div class="d-row-v10 row-target"><div class="w-type"><span class="tag-type">Target</span></div><div class="w-name" style="color:var(--red);">target</div><div class="w-desc" style="color:var(--arctic);"><b>Diagnóstico clínico de Riesgo Cardiovascular.</b></div><div class="w-vals"><span class="tag-pill">0: Sano</span><span class="tag-pill">1: Riesgo</span></div></div>
</div>
"""


# 2. EJECUCIÓN FINAL UNIFICADA
if 'df' in locals():
    reg, var = df.shape

    # Inyectamos todo en un solo display
    display(HTML(f2_body_template.format(
        styles=f2_styles_v38,
        URL_AUTOR="https://github.com",
        URL_INDEX="#",
        reg=f"{reg:,}",
        var=var,
        table_preview=render_v38(df.head(), "Vista Previa de Registros", "F2.2-PREVIEW"),
        table_stats=render_v38(df.describe().round(2), "Distribución Estadística Global", "F2.2-STATS"),
        diccionario=dict_v10_final # Tu variable con el HTML del diccionario
    )))

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1
,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.0,1000.00,1000.00,1000.00,1000.00
mean,5048704.41,49.24,0.76,0.98,151.75,311.45,0.30,0.75,145.48,0.5,2.71,1.54,1.22,0.58
std,2895904.50,17.86,0.42,0.95,29.97,132.44,0.46,0.77,34.19,0.5,1.72,1.00,0.98,0.49
min,103368.00,20.00,0.00,0.00,94.00,0.00,0.00,0.00,71.00,0.0,0.00,0.00,0.00,0.00
